<a href="https://colab.research.google.com/github/Shavina-143/CRM-CW1_w2083673/blob/main/CWK1_SP500_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

start = '2021-01-01'
end = '2025-01-01'  # End date excluded

In [ ]:
# Extract the S&P 500 company list from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

table = pd.read_html(
    url,
    storage_options={"User-Agent": "Mozilla/5.0"}
)

sp500_df = table[0].copy()

# Keep the original symbols and create Yahoo-compatible tickers
sp500_df['Yahoo_Ticker'] = sp500_df['Symbol'].str.replace(
    '.', '-', regex=False
)

tickers = sp500_df['Yahoo_Ticker'].tolist()

# Record when the list was retrieved
retrieval_date = pd.Timestamp.now(tz='UTC').isoformat()
sp500_df['Retrieved_UTC'] = retrieval_date

# Save the constituent-list snapshot
sp500_df.to_csv('sp500_constituents.csv', index=False)

print('Retrieval time (UTC):', retrieval_date)
print('Total tickers extracted:', len(tickers))

sp500_df[
    ['Symbol', 'Yahoo_Ticker', 'Security', 'GICS Sector']
].head(10)

Retrieval time (UTC): 2026-11-07T04:30:40.097076+00:00
Total tickers extracted: 503


,Symbol,Yahoo_Ticker,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology
5,ADBE,ADBE,Adobe Inc.,Information Technology
6,AMD,AMD,Advanced Micro Devices,Information Technology
7,AES,AES,AES Corporation,Utilities
8,AFL,AFL,Aflac,Financials
9,A,A,Agilent Technologies,Health Care


In [ ]:
# Download shares and the S&P 500 benchmark
all_tickers = tickers + ['^GSPC']

raw_data = yf.download(
    all_tickers,
    start=start,
    end=end,
    interval='1d',
    auto_adjust=False,
    actions=True,
    keepna=True,
    group_by='column',
    threads=4
)

if raw_data is None or raw_data.empty:
    raise ValueError('No data downloaded. Check the download messages.')

# Preserve the full download before cleaning
raw_data.to_csv('market_data_raw_2021_2024.csv')

print('Download time (UTC):', pd.Timestamp.now(tz='UTC').isoformat())

# Adjusted closing prices: same table structure as the tutorial
df = raw_data['Adj Close'].reindex(columns=tickers)

# Keep benchmark prices separately
market_prices = raw_data['Adj Close'].reindex(columns=['^GSPC'])

# Initial download checks
no_price_data = df.columns[df.isna().all()].tolist()

print('Share price table shape:', df.shape)
print('Shares with some adjusted-price data:', df.notna().any().sum())
print('Tickers with no adjusted-price data:', no_price_data)
print('Benchmark price observations:', market_prices['^GSPC'].notna().sum())

# Display a small sample
raw_data['Adj Close'].reindex(
    columns=['AAPL', 'MSFT', '^GSPC']
).head()

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['VYLR', 'HONA', 'FDXF', 'Q', 'SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Download time (UTC): 2026-11-07T04:34:53.593193+00:00
Share price table shape: (1005, 503)
Shares with some adjusted-price data: 498
Tickers with no adjusted-price data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Benchmark price observations: 1005


Ticker,AAPL,MSFT,^GSPC
Date,,,
2021-01-04,125.632507,207.565399,3700.649902
2021-01-05,127.185730,207.765610,3726.860107
2021-01-06,122.904533,202.378403,3748.139893
2021-01-07,127.098434,208.137466,3803.790039
2021-01-08,128.195404,209.405609,3824.679932


In [ ]:
# Check duplicate dates before aligning the tables
if raw_data.index.duplicated().any():
    raise ValueError('Duplicate dates found. Investigate before continuing.')

# Use dates with available benchmark prices
benchmark_dates = market_prices.index[
    market_prices['^GSPC'].notna()
]

# Align the required fields to those dates
df = raw_data['Adj Close'].reindex(
    index=benchmark_dates, columns=tickers
)

close_prices = raw_data['Close'].reindex(
    index=benchmark_dates, columns=tickers
)

volumes = raw_data['Volume'].reindex(
    index=benchmark_dates, columns=tickers
)

# Create one validation record per share
validation = pd.DataFrame(index=tickers)
validation.index.name = 'Share'

validation['Price_days'] = df.notna().sum()
validation['Missing_adj_close'] = df.isna().sum()
validation['Missing_close'] = close_prices.isna().sum()
validation['Missing_volume'] = volumes.isna().sum()

validation['Nonpositive_price_days'] = (
    (df <= 0) | (close_prices <= 0)
).sum()

validation['Negative_volume_days'] = (volumes < 0).sum()
validation['Zero_volume_days'] = (volumes == 0).sum()

validation['First_price_date'] = df.apply(
    lambda column: column.first_valid_index()
)
validation['Last_price_date'] = df.apply(
    lambda column: column.last_valid_index()
)

# Flag records needing investigation
check_columns = [
    'Missing_adj_close', 'Missing_close', 'Missing_volume',
    'Nonpositive_price_days', 'Negative_volume_days',
    'Zero_volume_days'
]

needs_review = validation[check_columns].gt(0).any(axis=1)

validation['Status'] = 'Pass initial numeric checks'
validation.loc[needs_review, 'Status'] = 'Needs review'
validation.loc[
    validation['Price_days'] == 0, 'Status'
] = 'No adjusted-price history returned'

# Save the validation evidence
validation.to_csv('share_validation_initial.csv')

print('Benchmark trading dates:', len(benchmark_dates))
print(validation['Status'].value_counts())

# Show shares needing investigation
validation.loc[
    validation['Status'] != 'Pass initial numeric checks'
].sort_values('Price_days').head(30)

Benchmark trading dates: 1005
Status
Pass initial numeric checks           486
Needs review                           12
No adjusted-price history returned      5
Name: count, dtype: int64


,Price_days,Missing_adj_close,Missing_close,Missing_volume,Nonpositive_price_days,Negative_volume_days,Zero_volume_days,First_price_date,Last_price_date,Status
Share,,,,,,,,,,
FDXF,0,1005,1005,1005,0,0,0,NaT,NaT,No adjusted-price history returned
SNDK,0,1005,1005,1005,0,0,0,NaT,NaT,No adjusted-price history returned
Q,0,1005,1005,1005,0,0,0,NaT,NaT,No adjusted-price history returned
HONA,0,1005,1005,1005,0,0,0,NaT,NaT,No adjusted-price history returned
VYLR,0,1005,1005,1005,0,0,0,NaT,NaT,No adjusted-price history returned
GEV,193,812,812,812,0,0,0,2024-03-27,2024-12-31,Needs review
SOLV,194,811,811,811,0,0,0,2024-03-26,2024-12-31,Needs review
RDDT,197,808,808,808,0,0,0,2024-03-21,2024-12-31,Needs review
VLTO,313,692,692,692,0,0,0,2023-10-04,2024-12-31,Needs review


In [ ]:
from IPython.display import display

# Inspect zero-volume dates
volume_checks = []

for share in ['FERG', 'SW']:
    v = volumes[share]
    zero_dates = v.index[v.eq(0)]
    positive_dates = v.index[v.gt(0)]

    volume_checks.append({
        'Share': share,
        'Zero_days_total': int(v.eq(0).sum()),
        'First_zero_date': zero_dates.min(),
        'Last_zero_date': zero_dates.max(),
        'First_positive_volume_date': positive_dates.min(),
        'Zero_days_last_80': int(v.tail(80).eq(0).sum()),
        'Missing_volume_last_80': int(v.tail(80).isna().sum())
    })

volume_review = pd.DataFrame(volume_checks).set_index('Share')
volume_review.to_csv('volume_review.csv')

print('Zero-volume investigation:')
display(volume_review)

# Count recorded corporate actions for each share
split_history = raw_data['Stock Splits'].reindex(columns=tickers)
dividend_history = raw_data['Dividends'].reindex(columns=tickers)

action_summary = pd.DataFrame({
    'Recorded_splits': split_history.gt(0).sum(),
    'Recorded_dividend_days': dividend_history.ne(0).where(
        dividend_history.notna(), False
    ).sum()
})

action_summary.index.name = 'Share'
action_summary.to_csv('corporate_action_summary.csv')

print('Shares with recorded splits:',
      int(action_summary['Recorded_splits'].gt(0).sum()))

display(
    action_summary.loc[
        action_summary['Recorded_splits'] > 0
    ].head(10)
)

Zero-volume investigation:


,Zero_days_total,First_zero_date,Last_zero_date,First_positive_volume_date,Zero_days_last_80,Missing_volume_last_80
Share,,,,,,
FERG,27,2021-01-07,2021-03-05,2021-01-04,0,0
SW,424,2021-01-05,2024-07-05,2021-01-04,0,0


Shares with recorded splits: 53


,Recorded_splits,Recorded_dividend_days
Share,,
MMM,1,16
GOOGL,1,3
GOOG,1,3
AMZN,1,0
APH,2,16
ANET,2,0
T,1,16
BDX,1,16
TECH,1,16


In [ ]:
# Daily returns without filling missing prices
adj_daily_returns = df.pct_change(fill_method=None)
close_daily_returns = close_prices.pct_change(fill_method=None)

# Previous session's values for comparison
previous_close = close_prices.shift(1)
previous_volume = volumes.shift(1)

# Identify recorded split events on benchmark trading dates
split_events = split_history.reindex(index=df.index).stack()
split_events = split_events[split_events > 0]

split_checks = []

for (date, share), ratio in split_events.items():
    split_checks.append({
        'Date': date,
        'Share': share,
        'Split_ratio': ratio,
        'Previous_close': previous_close.at[date, share],
        'Event_close': close_prices.at[date, share],
        'Adj_return_pct': 100 * adj_daily_returns.at[date, share],
        'Close_return_pct': 100 * close_daily_returns.at[date, share],
        'Previous_volume': previous_volume.at[date, share],
        'Event_volume': volumes.at[date, share]
    })

split_audit = pd.DataFrame(split_checks)

# Flag large moves or missing returns for investigation
# 30% is a screening threshold, not an automatic exclusion rule
split_audit['Review_flag'] = (
    split_audit['Adj_return_pct'].isna()
    | split_audit['Close_return_pct'].isna()
    | split_audit['Adj_return_pct'].abs().gt(30)
    | split_audit['Close_return_pct'].abs().gt(30)
)

split_audit['Absolute_adj_move_pct'] = (
    split_audit['Adj_return_pct'].abs()
)

split_audit.to_csv('split_event_audit.csv', index=False)

print('Recorded split events:', len(split_audit))
print('Events flagged for review:', split_audit['Review_flag'].sum())

# Show the ten largest adjusted-price movements
display(
    split_audit.sort_values(
        'Absolute_adj_move_pct', ascending=False
    ).head(10)
)

Recorded split events: 61
Events flagged for review: 0


,Date,Share,Split_ratio,Previous_close,Event_close,Adj_return_pct,Close_return_pct,Previous_volume,Event_volume,Review_flag,Absolute_adj_move_pct
20,2022-04-11,T,1.324,18.232628,19.629999,7.664126,7.664124,73121078.0,157485300,False,7.664126
22,2022-06-13,DXCM,4.000,73.462502,68.059998,-7.354097,-7.354097,3879200.0,3286900,False,7.354097
43,2024-04-01,MMM,1.196,88.687294,94.019997,6.012933,6.012927,5245776.0,13004800,False,6.012933
31,2023-01-04,GE,1.281,52.943974,56.025539,5.820418,5.820428,10280781.0,21031104,False,5.820418
38,2024-01-03,FLEX,1.327,22.833460,23.740000,3.970226,3.970226,7733358.0,17277700,False,3.970226
30,2022-11-30,TECH,4.000,81.767502,84.989998,3.941071,3.941047,812800.0,1992900,False,3.941071
16,2022-02-02,EXC,1.402,41.248219,42.860001,3.907522,3.907519,10562948.0,29696500,False,3.907522
3,2021-06-28,CSGP,10.000,88.264999,84.919998,-3.789726,-3.789726,9020000.0,3357000,False,3.789726
12,2021-11-02,DELL,1.973,56.517994,54.610001,-3.375906,-3.375904,6635002.0,18238300,False,3.375906
59,2024-12-16,PANW,2.000,196.559998,202.500000,3.021979,3.021979,3848800.0,6087000,False,3.021979


In [ ]:
# Identify shares that returned no adjusted prices
retry_tickers = validation.index[
    validation['Price_days'] == 0
].tolist()

print('Retrying:', retry_tickers)

retry_data = yf.download(
    retry_tickers,
    start=start,
    end=end,
    interval='1d',
    auto_adjust=False,
    actions=True,
    keepna=True,
    group_by='column',
    threads=False
)

retry_time = pd.Timestamp.now(tz='UTC').isoformat()

# Default to zero observations unless prices are returned
retry_counts = pd.Series(0, index=retry_tickers, dtype='int64')

if retry_data is not None:
    retry_data.to_csv('failed_tickers_retry_raw.csv')

    if not retry_data.empty:
        retry_counts = (
            retry_data['Adj Close']
            .reindex(columns=retry_tickers)
            .notna()
            .sum()
        )

# Save a record of the retry and its outcome
retry_log = retry_counts.rename('Adj_close_days').to_frame()
retry_log.index.name = 'Share'

retry_log['Attempted_UTC'] = retry_time
retry_log['Request_start'] = start
retry_log['Request_end_exclusive'] = end

retry_log['Result'] = np.where(
    retry_log['Adj_close_days'] > 0,
    'Data returned - needs validation',
    'No adjusted-price data returned'
)

retry_log.to_csv('download_retry_log.csv')

display(retry_log)

Retrying: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


[*********************100%***********************]  5 of 5 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['SNDK', 'FDXF', 'VYLR', 'HONA', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


,Adj_close_days,Attempted_UTC,Request_start,Request_end_exclusive,Result
Share,,,,,
FDXF,0,2026-11-07T04:51:00.462671+00:00,2021-01-01,2025-01-01,No adjusted-price data returned
HONA,0,2026-11-07T04:51:00.462671+00:00,2021-01-01,2025-01-01,No adjusted-price data returned
Q,0,2026-11-07T04:51:00.462671+00:00,2021-01-01,2025-01-01,No adjusted-price data returned
SNDK,0,2026-11-07T04:51:00.462671+00:00,2021-01-01,2025-01-01,No adjusted-price data returned
VYLR,0,2026-11-07T04:51:00.462671+00:00,2021-01-01,2025-01-01,No adjusted-price data returned


In [ ]:
# Create a documented selection log
selection_log = validation.copy()

# Add checks for infinite values
selection_log['Infinite_price_days'] = (
    np.isinf(df) | np.isinf(close_prices)
).sum()

selection_log['Infinite_volume_days'] = np.isinf(volumes).sum()

selection_log['Reason'] = ''

# Full daily coverage is our chosen comparability rule
coverage_columns = [
    'Missing_adj_close', 'Missing_close', 'Missing_volume'
]

incomplete = selection_log[coverage_columns].gt(0).any(axis=1)

selection_log.loc[incomplete, 'Reason'] = (
    'Incomplete daily coverage under the chosen full-period rule'
)

# Identify invalid numeric observations
invalid_columns = [
    'Nonpositive_price_days', 'Negative_volume_days',
    'Infinite_price_days', 'Infinite_volume_days'
]

invalid = selection_log[invalid_columns].gt(0).any(axis=1)

selection_log.loc[invalid, 'Reason'] = (
    'Invalid price or volume observations requiring resolution'
)

# Record the unsuccessful initial download and retry
selection_log.loc[
    selection_log['Price_days'] == 0, 'Reason'
] = 'No adjusted-price history returned in initial download or retry'

# Document unresolved historical continuity
selection_log.loc[['FERG', 'SW'], 'Reason'] = (
    'Unresolved earlier price/volume continuity around listing changes'
)

selection_log['Decision'] = np.where(
    selection_log['Reason'].eq(''),
    'Retain for features',
    'Exclude'
)

# Keep the original downloaded tables unchanged
candidate_tickers = selection_log.index[
    selection_log['Decision'] == 'Retain for features'
].tolist()

prices_for_analysis = df[candidate_tickers].copy()
close_for_analysis = close_prices[candidate_tickers].copy()
volume_for_analysis = volumes[candidate_tickers].copy()

# Save selection decisions and company information
selection_log.to_csv('share_selection_log.csv')

candidate_shares = sp500_df[
    sp500_df['Yahoo_Ticker'].isin(candidate_tickers)
].copy()

candidate_shares.to_csv('candidate_shares.csv', index=False)

print(selection_log['Decision'].value_counts())
print('Candidate price table:', prices_for_analysis.shape)

display(
    selection_log.loc[
        selection_log['Decision'] == 'Exclude',
        ['Price_days', 'Decision', 'Reason']
    ]
)

Decision
Retain for features    486
Exclude                 17
Name: count, dtype: int64
Candidate price table: (1005, 486)


,Price_days,Decision,Reason
Share,,,
APP,935,Exclude,Incomplete daily coverage under the chosen ful...
COIN,936,Exclude,Incomplete daily coverage under the chosen ful...
CEG,742,Exclude,Incomplete daily coverage under the chosen ful...
EXE,979,Exclude,Incomplete daily coverage under the chosen ful...
FDXF,0,Exclude,No adjusted-price history returned in initial ...
FERG,1005,Exclude,Unresolved earlier price/volume continuity aro...
GEHC,513,Exclude,Incomplete daily coverage under the chosen ful...
GEV,193,Exclude,Incomplete daily coverage under the chosen ful...
HONA,0,Exclude,No adjusted-price history returned in initial ...


In [ ]:
# Same table orientation used in the tutorial
df1 = prices_for_analysis.T

def calculate_weekly_returns(df1, shares=None):
    prices = df1 if shares is None else df1.loc[shares]

    prices = prices.T.copy()
    prices.index = pd.to_datetime(prices.index)
    prices = prices.sort_index()

    # Final available trading price in each Friday-ending week
    weekly_prices = prices.resample('W-FRI').last()

    # Keep weekly bins fully inside the study period
    week_starts = weekly_prices.index - pd.Timedelta(days=6)

    complete_weeks = (
        (week_starts >= pd.Timestamp(start))
        & (weekly_prices.index < pd.Timestamp(end))
    )

    weekly_prices = weekly_prices.loc[complete_weeks]

    # Remove only the first undefined return
    return weekly_prices.pct_change(fill_method=None).iloc[1:]


# Check benchmark coverage before resampling
benchmark_daily = market_prices.reindex(
    index=prices_for_analysis.index
)

if (
    not np.isfinite(benchmark_daily.to_numpy()).all()
    or benchmark_daily.le(0).any().any()
):
    raise ValueError('Benchmark prices need investigation.')

# Calculate share and benchmark returns using the same function
weekly_returns = calculate_weekly_returns(df1)

market_weekly_returns = calculate_weekly_returns(
    benchmark_daily.T
)

# Verify matching week labels and valid returns
if not weekly_returns.index.equals(market_weekly_returns.index):
    raise ValueError('Share and benchmark weeks do not match.')

if (
    not np.isfinite(weekly_returns.to_numpy()).all()
    or not np.isfinite(market_weekly_returns.to_numpy()).all()
):
    raise ValueError('Missing or infinite weekly returns found.')

# Arithmetic mean: keep decimal units for later calculations
mean_weekly_returns = weekly_returns.mean(axis=0).rename(
    'Mean_weekly_return'
)

# Save calculation evidence
weekly_returns.to_csv('weekly_returns_2021_2024.csv')
market_weekly_returns.to_csv('benchmark_weekly_returns.csv')
mean_weekly_returns.to_csv('mean_weekly_returns.csv')

print('Weekly return table:', weekly_returns.shape)
print('First return week:', weekly_returns.index.min().date())
print('Last return week:', weekly_returns.index.max().date())

# Display percentages for readability
display(
    mean_weekly_returns.mul(100)
    .rename('Mean_weekly_return_pct')
    .to_frame()
    .head(10)
)

Weekly return table: (207, 486)
First return week: 2021-01-15
Last return week: 2024-12-27


,Mean_weekly_return_pct
Ticker,
MMM,0.114247
AOS,0.199066
ABT,0.097133
ABBV,0.374916
ACN,0.237638
ADBE,0.099002
AMD,0.362408
AES,-0.141980
AFL,0.500423


In [ ]:
def calculate_weekly_beta(weekly_returns, market_returns, min_weeks=52):
    if min_weeks < 2:
        raise ValueError('At least two weekly observations are required.')

    results = []

    for share in weekly_returns.columns:
        # Match share and benchmark returns by week
        paired = pd.concat(
            [
                weekly_returns[share].rename('share'),
                market_returns.rename('market')
            ],
            axis=1
        ).replace([np.inf, -np.inf], np.nan).dropna()

        market_variance = paired['market'].var(ddof=1)
        beta = np.nan

        if len(paired) >= min_weeks and market_variance > 0:
            beta = (
                paired['share'].cov(paired['market'])
                / market_variance
            )

        results.append({
            'Share': share,
            'Beta': beta,
            'Paired_weeks': len(paired)
        })

    return pd.DataFrame(results).set_index('Share')


# Require every retained weekly return for each share
beta_df = calculate_weekly_beta(
    weekly_returns,
    market_weekly_returns['^GSPC'],
    min_weeks=len(weekly_returns)
)

if not np.isfinite(beta_df['Beta'].to_numpy()).all():
    raise ValueError('Some beta estimates need investigation.')

# Exactly two clustering features; observation counts stay separate
risk_return_features = pd.concat(
    [mean_weekly_returns, beta_df['Beta']],
    axis=1
)

risk_return_features.index.name = 'Share'

beta_df.to_csv('weekly_beta_results.csv')
risk_return_features.to_csv('risk_return_features.csv')

print('Risk/return feature table:', risk_return_features.shape)
print('Minimum paired weeks:', beta_df['Paired_weeks'].min())
print('Maximum paired weeks:', beta_df['Paired_weeks'].max())

# Display return percentages alongside beta and observation counts
preview = risk_return_features.join(beta_df['Paired_weeks']).copy()
preview['Mean_weekly_return_pct'] = (
    preview['Mean_weekly_return'] * 100
)

display(
    preview[
        ['Mean_weekly_return_pct', 'Beta', 'Paired_weeks']
    ].head(10)
)

Risk/return feature table: (486, 2)
Minimum paired weeks: 207
Maximum paired weeks: 207


,Mean_weekly_return_pct,Beta,Paired_weeks
Share,,,
MMM,0.114247,0.869079,207
AOS,0.199066,1.042805,207
ABT,0.097133,0.583543,207
ABBV,0.374916,0.425366,207
ACN,0.237638,1.143300,207
ADBE,0.099002,1.553525,207
AMD,0.362408,1.877622,207
AES,-0.141980,1.143912,207
AFL,0.500423,0.707197,207


In [ ]:
# Use the common daily trading dates
rfm_volume = volume_for_analysis.sort_index()
rfm_close = close_for_analysis.reindex(
    index=rfm_volume.index,
    columns=rfm_volume.columns
)

if len(rfm_volume) < 80:
    raise ValueError('Need 60 sessions plus 20 preceding sessions.')

observation_date = rfm_volume.index[-1]
latest_sessions = rfm_volume.index[-60:]

# Previous 20 sessions only: exclude the day being tested
average_volume = (
    rfm_volume.shift(1)
    .rolling(window=20, min_periods=20)
    .mean()
)

volume_threshold = 2 * average_volume

# Strictly greater than the threshold
high_volume_flags = (
    rfm_volume.gt(volume_threshold)
    & average_volume.notna()
)

if average_volume.loc[latest_sessions].isna().any().any():
    raise ValueError('Incomplete volume baselines in the RFM window.')

rfm_rows = []

for share in candidate_tickers:
    # Search the full valid study history for the latest event
    event_dates = rfm_volume.index[high_volume_flags[share]]

    last_event_date = (
        event_dates[-1] if len(event_dates) > 0 else pd.NaT
    )

    recency = (
        (observation_date.date() - last_event_date.date()).days
        if pd.notna(last_event_date)
        else pd.NA
    )

    frequency = int(
        high_volume_flags.loc[latest_sessions, share].sum()
    )

    monetary = (
        rfm_close.loc[latest_sessions, share]
        * rfm_volume.loc[latest_sessions, share]
    ).sum(min_count=60)

    rfm_rows.append({
        'Share': share,
        'Observation_date': observation_date,
        'Last_high_volume_date': last_event_date,
        'Recency': recency,
        'Frequency': frequency,
        'Monetary': monetary
    })

rfm_results = pd.DataFrame(rfm_rows).set_index('Share')
rfm_results['Recency'] = rfm_results['Recency'].astype('Int64')

# Exactly three clustering features; retain dates separately
rfm_features = rfm_results[
    ['Recency', 'Frequency', 'Monetary']
].copy()

rfm_results.to_csv('rfm_calculation_results.csv')
rfm_features.to_csv('rfm_features.csv')

print('Observation date:', observation_date.date())
print('60-session window starts:', latest_sessions[0].date())
print('Sessions in window:', len(latest_sessions))
print('RFM feature table:', rfm_features.shape)
print('Shares with no qualifying event:',
      rfm_features['Recency'].isna().sum())

display(rfm_results.head(10))

Observation date: 2024-12-31
60-session window starts: 2024-10-07
Sessions in window: 60
RFM feature table: (486, 3)
Shares with no qualifying event: 0


,Observation_date,Last_high_volume_date,Recency,Frequency,Monetary
Share,,,,,
MMM,2024-12-31,2024-12-20,11,4,2.599146e+10
AOS,2024-12-31,2024-12-20,11,2,4.748980e+09
ABT,2024-12-31,2024-12-20,11,1,3.523027e+10
ABBV,2024-12-31,2024-12-20,11,3,6.837054e+10
ACN,2024-12-31,2024-12-20,11,2,5.199430e+10
ADBE,2024-12-31,2024-12-13,18,3,1.030403e+11
AMD,2024-12-31,2024-12-13,18,4,3.131659e+11
AES,2024-12-31,2024-11-07,54,3,1.119578e+10
AFL,2024-12-31,2024-12-20,11,2,1.328456e+10


In [ ]:
example_share = 'MMM'

example = pd.DataFrame({
    'Close': rfm_close[example_share],
    'Volume': rfm_volume[example_share],
    'Previous_20_mean': average_volume[example_share],
    'Threshold': volume_threshold[example_share],
    'High_volume': high_volume_flags[example_share]
})

example['Daily_traded_value'] = example['Close'] * example['Volume']

# Independently identify events using explicit preceding-session slices
checked_events = []

for position in range(20, len(example)):
    previous_20 = example['Volume'].iloc[position - 20:position]
    checked_threshold = 2 * previous_20.mean()

    if example['Volume'].iloc[position] > checked_threshold:
        checked_events.append(example.index[position])

checked_last_event = checked_events[-1]

checked_recency = (
    observation_date.date() - checked_last_event.date()
).days

checked_frequency = sum(
    date in latest_sessions for date in checked_events
)

recent_example = example.loc[latest_sessions].copy()

checked_monetary = sum(
    float(close) * float(volume)
    for close, volume in zip(
        recent_example['Close'], recent_example['Volume']
    )
)

# Compare with the original calculations
assert checked_recency == rfm_features.loc[example_share, 'Recency']
assert checked_frequency == rfm_features.loc[example_share, 'Frequency']

assert np.isclose(
    checked_monetary,
    rfm_features.loc[example_share, 'Monetary'],
    rtol=1e-12,
    atol=0.01
)

# Show the threshold calculation for the latest event
previous_20 = example.loc[
    example.index < checked_last_event, 'Volume'
].tail(20)

print('Worked example:', example_share)
print('Observation date:', observation_date.date())
print('Latest qualifying event:', checked_last_event.date())
print('Preceding baseline sessions:', len(previous_20))
print(f'Previous 20-session mean volume: {previous_20.mean():,.2f}')
print(f'High-volume threshold: {2 * previous_20.mean():,.2f}')
print(f'Event-day volume: {example.loc[checked_last_event, "Volume"]:,.0f}')
print('Checked Recency:', checked_recency)
print('Checked Frequency:', checked_frequency)
print(f'Checked Monetary: ${checked_monetary:,.2f}')

recent_example.to_csv('MMM_worked_RFM_example.csv')

# These rows should explain the Frequency count
display(
    recent_example.loc[
        recent_example['High_volume'],
        ['Volume', 'Previous_20_mean', 'Threshold', 'Daily_traded_value']
    ]
)

Worked example: MMM
Observation date: 2024-12-31
Latest qualifying event: 2024-12-20
Preceding baseline sessions: 20
Previous 20-session mean volume: 3,027,540.00
High-volume threshold: 6,055,080.00
Event-day volume: 8,351,300
Checked Recency: 11
Checked Frequency: 4
Checked Monetary: $25,991,458,733.34


,Volume,Previous_20_mean,Threshold,Daily_traded_value
Date,,,,
2024-10-21,5940600,2829925.0,5659850.0,8.010305e+08
2024-10-22,13082300,2995035.0,5990070.0,1.723331e+09
2024-10-23,8116200,3491885.0,6983770.0,1.038143e+09
2024-12-20,8351300,3027540.0,6055080.0,1.079656e+09


In [ ]:
feature_columns = [
    'Mean_weekly_return', 'Beta', 'Recency', 'Frequency', 'Monetary'
]

# Combine the five calculated features by share
feature_table = risk_return_features.join(
    rfm_features,
    how='inner',
    validate='one_to_one'
)

# Check that no shares disappeared during the join
assert set(feature_table.index) == set(candidate_tickers)
assert np.isfinite(
    feature_table[feature_columns].to_numpy(dtype=float)
).all()

# Add company information and calculation metadata
company_info = candidate_shares.set_index('Yahoo_Ticker')[
    ['Security', 'GICS Sector']
]

feature_table = feature_table.join(
    company_info, validate='one_to_one'
)

feature_table = feature_table.join(
    rfm_results[['Observation_date', 'Last_high_volume_date']],
    validate='one_to_one'
)

feature_table['Paired_weeks'] = beta_df['Paired_weeks']
feature_table.index.name = 'Share'

assert feature_table[['Security', 'GICS Sector']].notna().all().all()

# Finalise the common sample under our chosen validation rules
selection_log.loc[feature_table.index, 'Decision'] = 'Include'
selection_log.loc[feature_table.index, 'Reason'] = (
    'Passed chosen coverage rules and five-feature completeness checks'
)

selection_log.to_csv('share_selection_log.csv')
feature_table.to_csv('share_feature_table.csv')

# Summaries in the original calculation units
feature_summary = feature_table[feature_columns].describe().T
feature_summary.to_csv('feature_summary.csv')

print('Shares in the common analysis sample:', len(feature_table))
print('Missing feature values:',
      feature_table[feature_columns].isna().sum().sum())

display(feature_summary)

Shares in the common analysis sample: 486
Missing feature values: 0


,count,mean,std,min,25%,50%,75%,max
Mean_weekly_return,486.0,0.002857,0.002652,-0.004468,0.001182,0.002619,0.004207,0.017772
Beta,486.0,0.973695,0.407339,0.055453,0.6666,0.950609,1.179538,3.77856
Recency,486.0,20.407407,21.342438,4.0,11.0,11.0,14.75,222.0
Frequency,486.0,2.989712,1.3308,0.0,2.0,3.0,4.0,8.0
Monetary,486.0,39404191889.20784,129138833232.971695,1297482617.047882,10195173392.27047,16683821504.623032,32276893043.872356,1847697379221.842773


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from importlib.metadata import version
from google.colab import files

# Record versions used for the calculations so far
packages = ['pandas', 'numpy', 'yfinance', 'matplotlib', 'seaborn']

pd.DataFrame({
    'Package': packages,
    'Version': [version(package) for package in packages]
}).to_csv('package_versions_task1.csv', index=False)

# Bundle the CSV files generated in this notebook workspace
with ZipFile('CWK1_Task1_backup.zip', 'w', ZIP_DEFLATED) as archive:
    for path in Path('.').glob('*.csv'):
        archive.write(path, arcname=path.name)

files.download('CWK1_Task1_backup.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>